# Phase 2 - Cleaning + Merge

Clean both raw datasets, aggregate crime to daily counts, then inner-join on date and save `merged.csv`.

In [1]:
import sys
sys.path.append("../src")
import pandas as pd
from cleaning import clean_crime, aggregate_crime_daily, clean_weather

In [ ]:
# only keep the cols I will use
cols = ["ID", "Date", "Primary Type", "Description", "Arrest", "Domestic", "Latitude", "Longitude"]
crime_raw = pd.read_csv("../data/raw/chicago_crime.csv", usecols=cols)
print("raw crime:", crime_raw.shape)

weather_raw = pd.read_csv("../data/raw/chicago_weather.csv", low_memory=False)
print("raw weather:", weather_raw.shape)

raw crime: (8544383, 8)
raw weather: (29056, 112)


## Crime cleaning

What I will do and why:

- **Parse the `Date` column to datetime** with the explicit `%m/%d/%Y %I:%M:%S %p` format. Letting pandas guess on 8.5M rows is painfully slow.
- **Drop rows missing date or category** - they can't be aggregated by day or by type, so they're useless for this project.
- **Dedupe on the city's `ID` column.** Same incident showing up twice would inflate counts.
- **Filter to 2018-2024** to match the project scope.
- **Uppercase `primary_type`** so groupbys are clean.
- **Cast `arrest`, `domestic` to bool** in case the city ever changes them to strings.

In [3]:
crime = clean_crime(crime_raw)
print("after cleaning:", crime.shape)
crime.head()

after cleaning: (1715802, 9)


,id,primary_type,description,arrest,domestic,latitude,longitude,date,date_only
0,13311263,OFFENSE INVOLVING CHILDREN,CHILD PORNOGRAPHY,True,False,NaN,NaN,2022-07-29 03:39:00,2022-07-29
1,13053066,NARCOTICS,MANUFACTURE / DELIVER - CRACK,True,False,NaN,NaN,2023-01-03 16:44:00,2023-01-03
2,12131221,ROBBERY,AGGRAVATED VEHICULAR HIJACKING,True,False,41.908418,-87.677407,2020-08-10 09:45:00,2020-08-10
4,13203321,CRIMINAL DAMAGE,TO VEHICLE,False,False,41.886018,-87.633938,2023-09-06 17:00:00,2023-09-06
5,13204489,THEFT,OVER $500,False,False,41.871835,-87.626151,2023-09-06 11:00:00,2023-09-06


In [4]:
# quick sanity check: top categories
crime["primary_type"].value_counts().head(10)

primary_type
THEFT                  382924
BATTERY                312737
CRIMINAL DAMAGE        190381
ASSAULT                146567
DECEPTIVE PRACTICE     126935
MOTOR VEHICLE THEFT    111970
OTHER OFFENSE          108143
ROBBERY                 62578
BURGLARY                60322
NARCOTICS               57787
Name: count, dtype: int64

## Aggregate to daily

One row per date with `total_crimes`, the six categories I care about, plus `arrest_count` and `domestic_count`. Other categories are still in the total but don't get their own column.

In [5]:
daily = aggregate_crime_daily(crime)
print("daily:", daily.shape)
daily.head()

daily: (2557, 10)


,date,total_crimes,theft,battery,burglary,assault,narcotics,criminal_damage,arrest_count,domestic_count
0,2018-01-01,1038,137,162,32,38,17,67,163,357
1,2018-01-02,564,128,89,20,28,25,46,121,110
2,2018-01-03,576,159,81,19,39,34,38,108,84
3,2018-01-04,601,134,102,31,42,22,47,111,115
4,2018-01-05,666,164,107,24,53,28,63,107,125


In [6]:
daily["total_crimes"].describe()

count    2557.000000
mean      671.021510
std       108.691916
min       341.000000
25%       604.000000
50%       676.000000
75%       743.000000
max      1901.000000
Name: total_crimes, dtype: float64

## Weather cleaning

Big things to remember here:

- NOAA stores `TMAX`, `TMIN` in **tenths of °C** and `PRCP`, `AWND` in **tenths of mm / m·s⁻¹**. Forgetting to divide by 10 is the classic bug. I will add Fahrenheit columns since the report will be in F.
- `SNOW`/`SNWD` are already in mm; NaN almost always means "no snow", so I fill with 0.
- For `TMAX`/`TMIN`, forward-fill up to 2 consecutive missing days. Longer gaps get dropped - I don't want to fabricate weather.
- For `AWND`, median-impute if missingness is under 5%.

In [ ]:
# look at missingness in our window before cleaning
wsub = weather_raw[(pd.to_datetime(weather_raw["DATE"]) >= "2018-01-01") &
                   (pd.to_datetime(weather_raw["DATE"]) <= "2024-12-31")]
wsub[["TMAX", "TMIN", "PRCP", "AWND", "SNOW", "SNWD"]].isna().mean().round(4)

TMAX    0.0
TMIN    0.0
PRCP    0.0
AWND    0.0
SNOW    0.0
SNWD    0.0
dtype: float64

In [ ]:
weather = clean_weather(weather_raw)
print("weather:", weather.shape)
weather.head()

weather: (2557, 9)


,date,tmax_c,tmin_c,tmax_f,tmin_f,prcp_mm,snow_mm,snwd_mm,awnd_ms
26013,2018-01-01,-17.1,-22.7,1.22,-8.86,0.0,0.0,30.0,4.8
26014,2018-01-02,-13.2,-22.7,8.24,-8.86,0.0,0.0,30.0,5.0
26015,2018-01-03,-8.2,-14.3,17.24,6.26,0.0,3.0,30.0,5.5
26016,2018-01-04,-11.0,-17.7,12.20,0.14,0.0,0.0,30.0,5.5
26017,2018-01-05,-11.6,-18.2,11.12,-0.76,0.0,0.0,30.0,4.8


In [ ]:
# sanity: tmax_f should be in [-40, 120]
weather[["tmax_f", "tmin_f", "prcp_mm", "awnd_ms"]].describe().round(2)

,tmax_f,tmin_f,prcp_mm,awnd_ms
count,2557.00,2557.00,2557.00,2557.00
mean,60.95,44.33,2.67,4.24
std,20.95,18.75,7.30,1.53
min,-9.76,-22.90,0.00,1.00
25%,42.98,30.20,0.00,3.10
50%,62.06,44.06,0.00,4.10
75%,80.06,62.06,1.30,5.10
max,100.04,80.96,89.70,12.50


## Merge + save

Inner join on `date` so I only keep days where both crime and weather are present. Then save to `data/processed/merged.csv`.

In [10]:
merged = daily.merge(weather, on="date", how="inner")
print("merged:", merged.shape)
merged.head()

merged: (2557, 18)


,date,total_crimes,theft,battery,burglary,assault,narcotics,criminal_damage,arrest_count,domestic_count,tmax_c,tmin_c,tmax_f,tmin_f,prcp_mm,snow_mm,snwd_mm,awnd_ms
0,2018-01-01,1038,137,162,32,38,17,67,163,357,-17.1,-22.7,1.22,-8.86,0.0,0.0,30.0,4.8
1,2018-01-02,564,128,89,20,28,25,46,121,110,-13.2,-22.7,8.24,-8.86,0.0,0.0,30.0,5.0
2,2018-01-03,576,159,81,19,39,34,38,108,84,-8.2,-14.3,17.24,6.26,0.0,3.0,30.0,5.5
3,2018-01-04,601,134,102,31,42,22,47,111,115,-11.0,-17.7,12.20,0.14,0.0,0.0,30.0,5.5
4,2018-01-05,666,164,107,24,53,28,63,107,125,-11.6,-18.2,11.12,-0.76,0.0,0.0,30.0,4.8


In [11]:
# confirm: no NaN in the columns we care about
print("NaN in total_crimes:", merged["total_crimes"].isna().sum())
print("NaN in tmax_f:", merged["tmax_f"].isna().sum())
print("NaN in tmin_f:", merged["tmin_f"].isna().sum())
print("date range:", merged["date"].min(), "->", merged["date"].max())

NaN in total_crimes: 0
NaN in tmax_f: 0
NaN in tmin_f: 0
date range: 2018-01-01 00:00:00 -> 2024-12-31 00:00:00


In [12]:
import os
os.makedirs("../data/processed", exist_ok=True)
merged.to_csv("../data/processed/merged.csv", index=False)
print("saved ../data/processed/merged.csv")

saved ../data/processed/merged.csv


Cleaned crime down to ~1.5M rows for 2018-2024, aggregated to one row per day, cleaned weather with the unit conversions and missing-value rules from the spec, and inner-joined to a single daily dataset (~2,500 rows). Saved to `data/processed/merged.csv` for the next phases.